In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import math
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit

In [ ]:
# ==========================================
# 1. ПІДГОТОВКА ДАНИХ (Lags)
# ==========================================
def create_lagged_features(df, target_col, exog_cols, lags=10):
    X, y = [], []
    data = df[[target_col] + exog_cols].values
    for i in range(lags, len(data)):
        # Сплющуємо попередні 10 кроків у один рядок (ознаки)
        X.append(data[i-lags:i].flatten())
        y.append(data[i, 0]) # Ціль — поточний CPU
    return np.array(X), np.array(y)

# TODO: Завантаж свої реальні файли
df_main = pd.read_csv("./clean_data/train_data_2.csv") 
df_ext = pd.read_csv("./clean_data/training_data.csv") 

# Параметри
target = 'cpu'
exog = ['rps', 'ram']
LAGS = 10

X_main, y_main = create_lagged_features(df_main, target, exog, lags=LAGS)
X_locust, y_locust = create_lagged_features(df_ext, target, exog, lags=LAGS)

In [ ]:




# ==========================================
# 2. COSINE SCHEDULER CALLBACK
# ==========================================
def cosine_scheduler(iteration, total_iterations, start_lr=0.1, min_lr=0.001):
    return min_lr + (start_lr - min_lr) * 0.5 * (1 + math.cos(math.pi * iteration / total_iterations))

class CosineLRScheduler(xgb.callback.TrainingCallback):
    def __init__(self, total_rounds, start_lr=0.1):
        self.total_rounds = total_rounds
        self.start_lr = start_lr

    def after_iteration(self, model, epoch, evals_log):
        # Оновлюємо learning_rate (eta) кожну ітерацію
        new_lr = cosine_scheduler(epoch, self.total_rounds, start_lr=self.start_lr)
        model.set_param('learning_rate', new_lr)
        return False

# ==========================================
# 3. КРОС-ВАЛІДАЦІЯ ТА ПОШУК (Tuning)
# ==========================================
print("🚀 Запуск XGBoost GridSearch з ітеративним навчанням...")
tscv = TimeSeriesSplit(n_splits=5)
ITERATIONS = 100 # Аналог епох

# Для спрощення зробимо перебір глибини дерева
best_mse = float('inf')
best_depth = 0

for depth in [3, 6, 9]:
    fold_mses = []
    for train_idx, val_idx in tscv.split(X_main):
        dtrain = xgb.DMatrix(X_main[train_idx], label=y_main[train_idx])
        dval = xgb.DMatrix(X_main[val_idx], label=y_main[val_idx])
        
        params = {'max_depth': depth, 'objective': 'reg:squarederror', 'eval_metric': 'rmse'}
        
        # Тренуємо з нашим косинусом
        bst = xgb.train(
            params, dtrain, 
            num_boost_round=ITERATIONS,
            callbacks=[CosineLRScheduler(ITERATIONS, start_lr=0.1)],
            evals=[(dval, 'validation')],
            verbose_eval=False
        )
        
        preds = bst.predict(dval)
        fold_mses.append(mean_squared_error(y_main[val_idx], preds))
    
    avg_mse = np.mean(fold_mses)
    print(f"Глибина {depth}: Середній MSE = {avg_mse:.5f}")
    if avg_mse < best_mse:
        best_mse = avg_mse
        best_depth = depth

# ==========================================
# 4. ФІНАЛЬНИЙ ТЕСТ (Train vs Val Loss)
# ==========================================
print(f"\n🏆 Найкраща глибина: {best_depth}. Тренуємо фінальну модель...")

dmain = xgb.DMatrix(X_main, label=y_main)
dlocust = xgb.DMatrix(X_locust, label=y_locust)

evals_result = {}
final_model = xgb.train(
    {'max_depth': best_depth, 'objective': 'reg:squarederror'},
    dmain, 
    num_boost_round=ITERATIONS,
    evals=[(dmain, 'train'), (dlocust, 'locust')],
    callbacks=[CosineLRScheduler(ITERATIONS)],
    evals_result=evals_result,
    verbose_eval=False
)

# ==========================================
# 5. ГРАФІКИ
# ==========================================
plt.figure(figsize=(12, 5))
plt.plot(evals_result['train']['rmse'], label='Train Loss (Standard)', color='blue')
plt.plot(evals_result['locust']['rmse'], label='Val Loss (Locust)', color='orange', linestyle='--')
plt.title(f'XGBoost Learning Curves (Cosine LR)\nBest Depth: {best_depth}', fontsize=14)
plt.xlabel('Boosting Rounds')
plt.ylabel('RMSE')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

# Прогноз на Locust
preds_locust = final_model.predict(dlocust)
plt.figure(figsize=(15, 5))
plt.plot(y_locust, label='Actual CPU', color='blue', alpha=0.6)
plt.plot(preds_locust, label='XGBoost Forecast', color='red', linestyle='--')
plt.title(f'XGBoost vs Locust Stress-Test (MSE: {mean_squared_error(y_locust, preds_locust):.5f})')
plt.legend()
plt.show()